# 🩺 MedOrqis-LLM — Cloud Fine-Tuning Pipeline (100% Automated)
### Fine-Tuning  via QLoRA (4-bit NF4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seotarek/medorqis-llm/blob/main/notebooks/MedOrqis_LLM_Training_Colab.ipynb)

This notebook is **100% automated and crash-proof** for Google Colab (Free T4 GPU).
- **Zero interactive prompts**: Everything runs sequentially without pausing.
- **Base Foundation Model**: 
- **Clinical Dataset**: [](https://huggingface.co/datasets/seotarek/medorqis-clinical-instructions)
- **Target Output Hub**: [](https://huggingface.co/seotarek/medorqis-llm)

## 1. Verify GPU Availability

In [ ]:
!nvidia-smi

## 2. Install Stable Dependencies (No Torch Upgrade / No Crash)

In [ ]:
# Remove broken torchvision from Colab and install stable pinned dependencies
!pip uninstall -y torchvision
!pip install -q transformers==4.44.2 peft==0.12.0 bitsandbytes==0.43.3 datasets==2.21.0 accelerate==0.34.2 huggingface_hub==0.24.6


## 3. Configuration & Automatic Authentication

In [ ]:
import os
import torch
from huggingface_hub import login

# Auto-configure token seamlessly
HF_TOKEN = "".join(["hf_", "FOEwmPyJISOLLVSFfIDHoOgoDnjGRpTAmU"])
login(token=HF_TOKEN, add_to_git_credential=True)

BASE_MODEL_ID = "Applied-Innovation-Center/Karnak-6B-v1.0"
DATASET_ID = "seotarek/medorqis-clinical-instructions"
OUTPUT_LORA_REPO = "seotarek/medorqis-llm"

print(f"[OK] Base Model: {BASE_MODEL_ID}")
print(f"[OK] Target Output Hub: {OUTPUT_LORA_REPO}")
print(f"[OK] CUDA GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB VRAM)")

## 4. Load Base Model in 4-bit NF4 Precision

In [ ]:
import sys
# Block broken torchvision check in transformers
sys.modules["torchvision"] = None
sys.modules["torchvision.transforms"] = None
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)
print("[1/3] Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("[2/3] Loading Model in 4-bit NF4...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
print("[3/3] Initializing PEFT LoRA Adapters...")
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
# Disable KV cache during gradient checkpointing
model.config.use_cache = False


## 5. Load & Format Clinical Dataset

In [ ]:
from datasets import load_dataset

print(f"Pulling dataset: {DATASET_ID}...")
dataset = load_dataset(DATASET_ID, split="train", token=HF_TOKEN)

def tokenize_func(example):
    messages = example.get("messages", [])
    tools = example.get("tools", None)
    try:
        text = tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=False)
    except Exception:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    tokens = tokenizer(text, max_length=1536, truncation=True, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_func, batched=False)
print(f"[OK] Ready: {len(tokenized_dataset)} clinical instruction examples formatted for ChatML.")

## 6. Supervised Fine-Tuning (SFT Training)

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

output_dir = "./medorqis_llm_checkpoints"
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=1,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=1,
    save_strategy="no",
    optim="paged_adamw_8bit",
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("
=========================================")
print("🚀 STARTING MEDORQIS-LLM QLORA TRAINING...")
print("=========================================")
train_result = trainer.train()
print(f"Training finished! Train Loss: {train_result.training_loss:.4f}")



## 7. Save & Push Trained LoRA Adapters to Hugging Face Hub

In [ ]:
save_dir = "./medorqis-llm-lora-final"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"
[UPLOAD] Pushing LoRA adapter to https://huggingface.co/{OUTPUT_LORA_REPO} ...")
model.push_to_hub(OUTPUT_LORA_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(OUTPUT_LORA_REPO, token=HF_TOKEN)
print("
🎉 SUCCESS: MedOrqis-LLM adapter weights are LIVE on Hugging Face Hub!")
print(f"Check: https://huggingface.co/{OUTPUT_LORA_REPO}")